用RotatE得到的embedding来做同等regression 和node2vec 以及full csv比较

In [21]:
import sys
from pathlib import Path

# 找到项目根目录（包含 src 的那一层）
p = Path.cwd()
while p != p.parent and not (p / "src").exists():
    p = p.parent

PROJECT_ROOT = p
sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("sys.path[0]:", sys.path[0])
print("src exists:", (PROJECT_ROOT / "src").exists())
print("target file exists:", (PROJECT_ROOT / "src" / "features" / "get_just_oncourt.py").exists())

from src.features.get_just_oncourt import get_oncourt_cols
print("import OK:", get_oncourt_cols)



import numpy as np
import pandas as pd




rot = pd.read_csv("../embeddings/rotate_L1B_cpu_player_embeddings.csv")

emb_cols = [c for c in rot.columns if c.startswith("e")]

# 把 complex 转成 numpy array
Z = rot[emb_cols].applymap(lambda x: complex(x)).to_numpy()

# 拆 real / imag
Z_re = np.real(Z)
Z_im = np.imag(Z)

# 构造新的 DataFrame（一次性 concat）
re_cols = [f"{c}_re" for c in emb_cols]
im_cols = [f"{c}_im" for c in emb_cols]

rot_feat = pd.concat(
    [
        rot[["player_id", "node_id"]],
        pd.DataFrame(Z_re, columns=re_cols),
        pd.DataFrame(Z_im, columns=im_cols),
    ],
    axis=1
)

print(rot_feat.shape)



cwd: c:\nba-salary-kg-project_newest\graph\notebooks
PROJECT_ROOT: c:\nba-salary-kg-project_newest
sys.path[0]: c:\nba-salary-kg-project_newest
src exists: True
target file exists: True
import OK: <function get_oncourt_cols at 0x0000024F8AF84220>
(991, 258)


C:\Users\Junha\AppData\Local\Temp\ipykernel_46892\144044021.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  Z = rot[emb_cols].applymap(lambda x: complex(x)).to_numpy()


In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

# === 1) Load tabular master table (the SAME one used in G1 Node2Vec regression) ===
# 你把这个路径改成你 G1 里读的那张表
TABULAR_PATH = Path("../../data/processed/training_level1_full.csv")   # <-- 这里按你实际改
# 备选：你可能是 parquet
# TABULAR_PATH = Path("../../data/processed/training_full_table.parquet")

if TABULAR_PATH.suffix == ".parquet":
    df = pd.read_parquet(TABULAR_PATH)
else:
    df = pd.read_csv(TABULAR_PATH)

print("tabular df shape:", df.shape)
print("columns contains:", {"player_id","season","log_salary"}.issubset(df.columns))

# === 2) On-court cols from your canonical function ===
oncourt_cols = get_oncourt_cols(df)
print("oncourt cols:", len(oncourt_cols))
print("sample:", oncourt_cols[:10])

tabular df shape: (2082, 132)
columns contains: True
oncourt cols: 81
sample: ['+/-', '3P%', '3PA', '3PA_per_gp', '3PA_per_min', '3PM', '3PM_per_gp', '3PM_per_min', 'AST', 'AST_TO_ratio']


In [24]:
ID_COLS = ["player_id", "season"]
TARGET_COL = "log_salary"

# 确保目标列存在
assert TARGET_COL in df.columns, f"Missing target: {TARGET_COL}"
assert all(c in df.columns for c in ID_COLS), f"Missing ID cols: {ID_COLS}"

# 只保留：ID + target + oncourt
base_df = df[ID_COLS + [TARGET_COL] + oncourt_cols].copy()

# 防止 nan 影响训练
base_df = base_df.dropna(subset=[TARGET_COL])

print("base_df shape:", base_df.shape)

# 额外 sanity：确认没有 award_/injury_ 被混进来
leaky_in_oncourt = [c for c in oncourt_cols if c.startswith(("award_", "injury_"))]
print("leaky in oncourt:", leaky_in_oncourt[:5], "count:", len(leaky_in_oncourt))
assert len(leaky_in_oncourt) == 0


base_df shape: (2082, 84)
leaky in oncourt: [] count: 0


In [25]:
# rot_feat 已经在你上面生成好了 (991, 258)
assert "player_id" in rot_feat.columns
assert rot_feat["player_id"].nunique() == len(rot_feat), "rot_feat player_id should be unique"

# merge
df_L1B = base_df.merge(
    rot_feat.drop(columns=["node_id"], errors="ignore"),
    on="player_id",
    how="inner"
)

print("df_L1B shape:", df_L1B.shape)
print("players before:", base_df["player_id"].nunique(), "after:", df_L1B["player_id"].nunique())


df_L1B shape: (2082, 340)
players before: 668 after: 668


In [26]:
TEST_SEASON = 2024

train = df_L1B[df_L1B["season"] < TEST_SEASON].copy()
test  = df_L1B[df_L1B["season"] == TEST_SEASON].copy()

print("train seasons:", sorted(train["season"].unique())[:5], "...", sorted(train["season"].unique())[-5:])
print("test seasons:", sorted(test["season"].unique()))
print("n_train:", len(train), "n_test:", len(test))

assert len(test) > 0, "Your TEST_SEASON has 0 rows; adjust TEST_SEASON."

train seasons: [2020, 2021, 2022, 2023] ... [2020, 2021, 2022, 2023]
test seasons: [2024]
n_train: 1659 n_test: 423


In [28]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def run_rf(train_df, test_df, label="log_salary"):
    X_train = train_df.drop(columns=["player_id", "season", label])
    y_train = train_df[label].astype(float)

    X_test = test_df.drop(columns=["player_id", "season", label])
    y_test = test_df[label].astype(float)

    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=20,
        min_samples_leaf=5,
        min_samples_split=10,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    return {
        "model": "RF",
        "R2": float(r2_score(y_test, pred)),
        "MAE": float(mean_absolute_error(y_test, pred)),
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "p": int(X_train.shape[1]),
    }

res_L1B_RF = run_rf(train, test)
res_L1B_RF


{'model': 'RF',
 'R2': 0.5207456331732341,
 'MAE': 0.6548237044776936,
 'n_train': 1659,
 'n_test': 423,
 'p': 337}

In [29]:
from sklearn.linear_model import Ridge

def run_ridge(train_df, test_df, label="log_salary", alpha=10.0):
    X_train = train_df.drop(columns=["player_id", "season", label])
    y_train = train_df[label].astype(float)

    X_test = test_df.drop(columns=["player_id", "season", label])
    y_test = test_df[label].astype(float)

    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("ridge", Ridge(alpha=alpha, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    return {
        "model": f"Ridge(alpha={alpha})",
        "R2": float(r2_score(y_test, pred)),
        "MAE": float(mean_absolute_error(y_test, pred)),
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "p": int(X_train.shape[1]),
    }

res_L1B_Ridge = run_ridge(train, test, alpha=10.0)
res_L1B_Ridge

{'model': 'Ridge(alpha=10.0)',
 'R2': 0.42473263686243046,
 'MAE': 0.701916202188051,
 'n_train': 1659,
 'n_test': 423,
 'p': 337}

In [30]:
results = pd.DataFrame([res_L1B_RF, res_L1B_Ridge])
results.insert(0, "setting", "L1-B: On-court + RotatE (Re+Im)")
results


,setting,model,R2,MAE,n_train,n_test,p
0,L1-B: On-court + RotatE (Re+Im),RF,0.520746,0.654824,1659,423,337
1,L1-B: On-court + RotatE (Re+Im),Ridge(alpha=10.0),0.424733,0.701916,1659,423,337
